# DEMAS — Portal de Dados Abertos do SUS

This notebook demonstrates the **DEMAS** accessor for the Brazilian Ministry of
Health's open data portal — 80+ public health datasets with **no API key required**.

**Coverage:** arboviroses (dengue, chikungunya, zika, febre amarela), vacinação
(PNI), CNES, SISAGUA, SISVAN, vigilância ambiental, saúde indígena, and more.

**Portal:** https://dadosabertos.saude.gov.br/  
**API:** https://apidadosabertos.saude.gov.br/  
**Swagger:** https://apidadosabertos.saude.gov.br/static/swagger.json

> **Note:** The API caps page size at 20 records and uses `offset` as a
> 0-indexed page number. `get_dataset()` fetches one page by default;
> use `get_dataset_all()` (with a `max_pages` guard) for full fetches.

In [1]:
from epidatasets.sources.demas import DemasAccessor

demas = DemasAccessor()
print(demas)

DEMAS — Portal de Dados Abertos do SUS (Brazilian Ministry of Health). 80+ public health datasets including arboviroses, vacinação, CNES, SISAGUA, vigilância ambiental and saúde indígena. (https://dadosabertos.saude.gov.br/)


## 1. Dataset discovery

The catalog is driven by the API's OpenAPI/Swagger specification.

In [2]:
domains = demas.list_domains()
print(f"{len(domains)} domains")
domains

INFO:epidatasets.sources.demas:Fetching https://apidadosabertos.saude.gov.br/static/swagger.json (attempt 1/3)


12 domains


,domain,description
0,Autenticação,Controle de Acesso da API
1,Agravo Arboviroses,Cadastro de ocorrência de agravo de arboviroses
2,Assistência à Saúde,
3,Atenção Primária,
4,Educação em Saúde,
5,BNAFAR,Base Nacional de Dados de Ações e Serviços da ...
6,Ciência & Tecnologia,
7,CNES,Cadastro Nacional de Estabelecimentos de Saúde
8,Macrorregião e Região de Saúde,Informações de Macrorregião e Região de Saúde
9,Plataforma Brasil,Cadastro de Projetos de Pesquisa na Plataforma...


In [3]:
datasets = demas.list_datasets()
print(f"{len(datasets)} datasets")
datasets.head(10)

80 datasets


,domain,endpoint,summary,has_year_filter,query_params
0,CNES,/cnes/tipounidades,Obtém todos os tipos de unidade.,False,
1,CNES,/cnes/tipounidades/{codigo_tipo_unidade},Obtém tipo de unidade utilizando o código do t...,False,
2,CNES,/cnes/estabelecimentos,Obtém todos os estabelecimentos.,False,"codigo_tipo_unidade, codigo_uf, codigo_municip..."
3,CNES,/cnes/estabelecimentos/{codigo_cnes},Obtém estabelecimento utilizando o código CNES.,False,
4,SISAGUA,/sisagua/vigilancia-parametros-basicos,Obtém lista de parâmetros básicos de vigilânci...,False,"uf, codigo_ibge, motivo_da_coleta, tipo_da_for..."
5,SISAGUA,/sisagua/controle-semestral,Obtém lista de parâmetros de controle semestral,False,"uf, codigo_ibge, cnpj_da_instituicao, tipo_da_..."
6,SISAGUA,/sisagua/controle-mensal-parametros-basicos,Obtém lista de parâmetros básicos de controle ...,False,"uf, codigo_ibge, cnpj_da_instituicao, tipo_da_..."
7,SISAGUA,/sisagua/pontos-de-captacao,Dados sobre os pontos de captação de água para...,False,"codigo_ibge, uf, limit, offset"
8,SISAGUA,/sisagua/cadastro-carro-pipa-populacao,Dados cadastrais sobre Carros-Pipa utilizados ...,False,"limit, offset"
9,SISAGUA,/sisagua/cadastro-carro-pipa-procedencia,SISAGUA - Cadastro Carro Pipa Procedência,False,"limit, offset"


## 2. Search by keyword

Search is case-insensitive across endpoint path, summary, and domain (Portuguese).

In [4]:
demas.search_datasets("dengue")

,domain,endpoint,summary,has_year_filter,query_params
0,Agravo Arboviroses,/arboviroses/dengue,Obtém base de ocorrência de arbovirose Dengue,True,"nu_ano, limit, offset"


In [5]:
demas.search_datasets("vacinação")

,domain,endpoint,summary,has_year_filter,query_params
0,Vacinação,/vacinacao/doses-aplicadas-pni-2020,Doses aplicadas pelo Programa de Nacional Imun...,False,"uf_estabelecimento, uf_paciente, limit, offset"
1,Vacinação,/vacinacao/doses-aplicadas-pni-2021,Doses aplicadas pelo Programa de Nacional Imun...,False,"uf_estabelecimento, limit, offset"
2,Vacinação,/vacinacao/doses-aplicadas-pni-2022,Doses aplicadas pelo Programa de Nacional Imun...,False,"uf_paciente, limit, offset"
3,Vacinação,/vacinacao/doses-aplicadas-pni-2023,Doses aplicadas pelo Programa de Nacional Imun...,False,"uf_estabelecimento, limit, offset"
4,Vacinação,/vacinacao/doses-aplicadas-pni-2024,Dose aplicadas pelo Programa Nacional de Imuni...,False,"limit, offset"
5,Vacinação,/vacinacao/doses-aplicadas-pni-2025,Dose aplicadas pelo Programa Nacional de Imuni...,False,"limit, offset"
6,Vacinação,/vacinacao/doses-aplicadas-pni-2026,Dose aplicadas pelo Programa Nacional de Imuni...,False,"limit, offset"
7,Vacinação,/vacinacao/esavi,Evento supostamente atribuível à vacinação ou ...,False,"limit, offset"
8,Vacinação,/vacinacao/sistema-de-informacao-de-insumos-es...,Monitoramento dos dados de doses distribuídas,False,"limit, offset"


## 3. Fetch a single page — Dengue arbovirose records

Each endpoint returns a dict with an endpoint-specific list key (e.g.
`{"parametros": [...]}`); the accessor auto-extracts the records.

In [6]:
dengue = demas.get_dataset("/arboviroses/dengue", year=2024)
print(f"{len(dengue)} records (one page, max 20)")
dengue[["id_agravo", "dt_notific", "sg_uf_not", "nu_ano", "cs_sexo", "classi_fin"]].head()

INFO:epidatasets.sources.demas:Fetching https://apidadosabertos.saude.gov.br/arboviroses/dengue (attempt 1/3)


20 records (one page, max 20)


,id_agravo,dt_notific,sg_uf_not,nu_ano,cs_sexo,classi_fin
0,A90,2024-12-31,29,2024,F,11
1,A90,2024-12-30,29,2024,F,10
2,A90,2024-12-30,29,2024,M,10
3,A90,2024-12-31,29,2024,M,8
4,A90,2024-12-30,29,2024,F,8


## 4. Convenience method — Chikungunya

In [7]:
chik = demas.get_arbovirose(disease="chikungunya", year=2024)
chik[["id_agravo", "dt_notific", "sg_uf_not", "classi_fin"]].head()

INFO:epidatasets.sources.demas:Fetching https://apidadosabertos.saude.gov.br/arboviroses/chikungunya (attempt 1/3)


,id_agravo,dt_notific,sg_uf_not,classi_fin
0,A92.0,2024-01-08,32,13
1,A92.0,2024-01-08,32,13
2,A92.0,2024-01-08,32,5
3,A92.0,2024-01-17,32,5
4,A92.0,2024-01-09,32,5


## 5. Vacinação — PNI doses applied (2024)

In [8]:
pni = demas.get_vacinacao_pni(year=2024)
pni.head()

INFO:epidatasets.sources.demas:Fetching https://apidadosabertos.saude.gov.br/vacinacao/doses-aplicadas-pni-2024 (attempt 1/3)


,descricao_natureza_estabelecimento,codigo_via_administracao,nome_pais_paciente,codigo_origem_registro,codigo_pais_paciente,nome_raca_cor_paciente,sigla_vacina,codigo_vacina_fabricante,data_vacina,codigo_condicao_maternal,...,descricao_vacina,descricao_origem_registro,data_entrada_rnds,descricao_vacina_categoria_atendimento,nome_etnia_indigena_paciente,tipo_sexo_paciente,descricao_nacionalidade_paciente,codigo_troca_documento,codigo_dose_vacina,descricao_dose_vacina
0,ADMINISTRACAO PUBLICA,0,BRASIL,None,10,PARDA,VPC10,None,2024-01-02,None,...,Vacina pneumo 10,None,2024-02-19,None,None,F,B,None,1,1ª Dose
1,ADMINISTRACAO PUBLICA,0,BRASIL,None,10,PARDA,VPC10,None,2024-01-16,None,...,Vacina pneumo 10,None,2024-01-24,None,None,F,B,None,2,2ª Dose
2,ADMINISTRACAO PUBLICA,0,BRASIL,None,10,PARDA,VPC10,None,2024-01-08,None,...,Vacina pneumo 10,None,2024-01-10,None,None,F,B,None,2,2ª Dose
3,ADMINISTRACAO PUBLICA,0,BRASIL,None,10,PRETA,VPC10,None,2024-01-24,None,...,Vacina pneumo 10,None,2024-01-30,None,None,F,B,None,1,1ª Dose
4,ADMINISTRACAO PUBLICA,0,BRASIL,None,10,PRETA,VPC10,None,2024-01-05,None,...,Vacina pneumo 10,None,2024-01-07,None,None,M,B,None,1,1ª Dose


## 6. Full pagination (with a page guard)

Dengue has millions of records. Use `max_pages` to bound the fetch.

In [9]:
dengue_all = demas.get_dataset_all("/arboviroses/dengue", year=2024, max_pages=10)
print(f"{len(dengue_all)} records fetched (10 pages x 20)")
dengue_all["sg_uf_not"].value_counts().head(10)

INFO:epidatasets.sources.demas:Loading cached data: /home/fccoelho/.cache/epi_data/demas/apidadosabertos.saude.gov.br_arboviroses_dengue_limit-20_nu_ano-2024_offset-0.json
INFO:epidatasets.sources.demas:Page 0: 20 records (total: 20)
INFO:epidatasets.sources.demas:Fetching https://apidadosabertos.saude.gov.br/arboviroses/dengue (attempt 1/3)
INFO:epidatasets.sources.demas:Page 1: 20 records (total: 40)
INFO:epidatasets.sources.demas:Fetching https://apidadosabertos.saude.gov.br/arboviroses/dengue (attempt 1/3)
INFO:epidatasets.sources.demas:Page 2: 20 records (total: 60)
INFO:epidatasets.sources.demas:Fetching https://apidadosabertos.saude.gov.br/arboviroses/dengue (attempt 1/3)
INFO:epidatasets.sources.demas:Page 3: 20 records (total: 80)
INFO:epidatasets.sources.demas:Fetching https://apidadosabertos.saude.gov.br/arboviroses/dengue (attempt 1/3)
INFO:epidatasets.sources.demas:Page 4: 20 records (total: 100)
INFO:epidatasets.sources.demas:Fetching https://apidadosabertos.saude.gov.br/

200 records fetched (10 pages x 20)


sg_uf_not
52    125
31     42
53     24
29      5
35      3
23      1
Name: count, dtype: int64

## 7. CNES — establishment lookup (path parameter)

Some endpoints take path parameters like `{codigo_cnes}`.

In [10]:
tipos = demas.get_dataset("/cnes/tipounidades")
print(f"{len(tipos)} unit types")
tipos.head()

INFO:epidatasets.sources.demas:Fetching https://apidadosabertos.saude.gov.br/cnes/tipounidades (attempt 1/3)


39 unit types


,codigo_tipo_unidade,descricao_tipo_unidade
0,80,LABORATORIO DE SAUDE PUBLICA
1,81,CENTRAL DE REGULACAO DO ACESSO
2,79,OFICINA ORTOPEDICA
3,82,"CENTRAL DE NOTIFICACAO,CAPTACAO E DISTRIB DE O..."
4,78,UNIDADE DE ATENCAO EM REGIME RESIDENCIAL


## 8. Using the registry

The accessor is reachable through the plugin registry.

In [11]:
from epidatasets import get_source, list_sources

print("demas" in list_sources())
demas2 = get_source("demas")
demas2.list_datasets(domain="Agravo Arboviroses")

INFO:epidatasets.sources.demas:Loading cached data: /home/fccoelho/.cache/epi_data/demas/swagger_spec.json


True


,domain,endpoint,summary,has_year_filter,query_params
0,Agravo Arboviroses,/arboviroses/zikavirus,Obtém base de ocorrência de arbovirose zikavirus,True,"nu_ano, limit, offset"
1,Agravo Arboviroses,/arboviroses/dengue,Obtém base de ocorrência de arbovirose Dengue,True,"nu_ano, limit, offset"
2,Agravo Arboviroses,/arboviroses/chikungunya,Obtém base de ocorrência de arbovirose Chikung...,True,"nu_ano, limit, offset"
3,Agravo Arboviroses,/arboviroses/febre-amarela-humanos-primatas-na...,Notificações de casos suspeitos da doença de f...,False,"limit, offset"
4,Agravo Arboviroses,/arboviroses/febre-amarela-epzootias,Ocorrências de epzootias de febre amarela por ...,False,"macrorreg_ocor, uf_ocor, mes_ocor, ano_ocor, l..."
